# Practical 4 — Autoencoder for Anomaly Detection

**Aim:** Build an autoencoder that learns to reconstruct normal examples and flags examples with high reconstruction error as anomalies.

**Demonstration dataset:** MNIST. For a simple demonstration, train only on digit zero and treat other digits as unusual examples. This is a teaching demonstration, not a production anomaly detector.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
print("TensorFlow:", tf.__version__)

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Train only on digit 0 (normal class)
x_train_normal = x_train[y_train == 0]
x_test_normal = x_test[y_test == 0]
x_test_anomaly = x_test[y_test != 0]
print("Normal training examples:", len(x_train_normal))
print("Normal test examples:", len(x_test_normal))
print("Anomaly test examples:", len(x_test_anomaly))

## Build encoder and decoder

The encoder compresses the input into a smaller latent representation. The decoder reconstructs the original image. A high reconstruction error can indicate an anomaly.

In [ ]:
input_img = keras.Input(shape=(28, 28))
x = layers.Flatten()(input_img)
x = layers.Dense(64, activation="relu")(x)
latent = layers.Dense(16, activation="relu", name="latent_space")(x)

x = layers.Dense(64, activation="relu")(latent)
x = layers.Dense(784, activation="sigmoid")(x)
output_img = layers.Reshape((28, 28))(x)

autoencoder = keras.Model(input_img, output_img)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

In [ ]:
history = autoencoder.fit(
    x_train_normal, x_train_normal,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    shuffle=True
)

In [ ]:
normal_recon = autoencoder.predict(x_test_normal, verbose=0)
anomaly_recon = autoencoder.predict(x_test_anomaly, verbose=0)

normal_error = np.mean(np.square(x_test_normal - normal_recon), axis=(1, 2))
anomaly_error = np.mean(np.square(x_test_anomaly - anomaly_recon), axis=(1, 2))

threshold = np.percentile(normal_error, 95)
print("Reconstruction error threshold:", threshold)
print("Average normal error:", normal_error.mean())
print("Average anomaly error:", anomaly_error.mean())

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(normal_error, bins=40, alpha=0.6, label="Normal (digit 0)")
plt.hist(anomaly_error, bins=40, alpha=0.6, label="Other digits")
plt.axvline(threshold, linestyle="--", label="Threshold")
plt.xlabel("Reconstruction error")
plt.ylabel("Count")
plt.title("Anomaly detection using autoencoder")
plt.legend()
plt.show()

In [ ]:
sample = x_test_anomaly[:5]
reconstructed = autoencoder.predict(sample, verbose=0)

plt.figure(figsize=(10, 4))
for i in range(5):
    plt.subplot(2, 5, i + 1)
    plt.imshow(sample[i], cmap="gray")
    plt.title("Original")
    plt.axis("off")
    plt.subplot(2, 5, i + 6)
    plt.imshow(reconstructed[i], cmap="gray")
    plt.title("Reconstructed")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Viva questions
1. What is an autoencoder? — A neural network trained to reconstruct its input.
2. What are encoder and decoder? — Encoder compresses input; decoder reconstructs it.
3. What is latent space? — The compressed representation learned by the encoder.
4. What is reconstruction error? — Difference between original input and reconstructed output.
5. How is anomaly detected? — A sample with reconstruction error above a chosen threshold is flagged.
6. Why train on normal data? — The model learns normal patterns and may reconstruct unusual patterns poorly.

**Conclusion:** Built an autoencoder and demonstrated anomaly detection using reconstruction error.